In [2]:
# 1 Prepare the virtual screening library

# Collect potential compounds for screening as SMILES.


In [3]:
!pip install rdkit

In [4]:
import pandas as pd
from rdkit import Chem

vs_data = pd.read_csv("VS_new_compounds.csv")  # Column: "SMILES"
vs_data.dropna(subset=['SMILES'], inplace=True)
print(vs_data)

              Common_Name                                             SMILES
0             Bivalirudin  CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...
1               Goserelin  CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2            Gramicidin D  CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3            Desmopressin  N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...
4              Cetrorelix  CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...
...                   ...                                                ...
12304       Tulmimetostat  COC1CN([C@H]2CC[C@H]([C@@]3(C)Oc4c(Cl)cc(C(=O)...
12305        Ibuzatrelvir  COC(=O)N[C@H](C(=O)N1C[C@H](C(F)(F)F)C[C@H]1C(...
12306        Cetyl oleate         CCCCCCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC
12307  Cetyl myristoleate             CCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC
12308  Cetyl palmitoleate           CCCCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC

[12309 rows x 2 columns]


In [5]:
## take only smile
smiles = vs_data[['SMILES']]
smiles

,SMILES
0,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...
1,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...
4,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...
...,...
12304,COC1CN([C@H]2CC[C@H]([C@@]3(C)Oc4c(Cl)cc(C(=O)...
12305,COC(=O)N[C@H](C(=O)N1C[C@H](C(F)(F)F)C[C@H]1C(...
12306,CCCCCCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC
12307,CCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC


In [6]:
#
# descriptor function to include physicochemical and topological descriptors

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski

# Function to compute extended descriptors
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        # Return NaN for invalid SMILES
        return pd.Series([np.nan]*11,
                         index=["MolWt", "LogP", "NumHDonors", "NumHAcceptors",
                                "NumRings", "NumRotatableBonds", "NumHeavyAtoms",
                                "NumHeteroatoms", "NumAromaticRings", "TPSA", "FractionCSP3"])

    # Compute descriptors
    return pd.Series([
        Descriptors.MolWt(mol),                  # Molecular Weight
        Crippen.MolLogP(mol),                    # LogP (hydrophobicity)
        Lipinski.NumHDonors(mol),                # H-bond donors
        Lipinski.NumHAcceptors(mol),             # H-bond acceptors
        Lipinski.RingCount(mol),                 # Number of rings
        Lipinski.NumRotatableBonds(mol),         # Rotatable bonds
        Descriptors.HeavyAtomCount(mol),         # Number of heavy atoms
        Descriptors.NumHeteroatoms(mol),         # Number of heteroatoms
        Lipinski.NumAromaticRings(mol),          # Number of aromatic rings
        Descriptors.TPSA(mol),                   # Topological Polar Surface Area
        Lipinski.FractionCSP3(mol)               # Fraction of sp3 carbons
    ], index=["MolWt", "LogP", "NumHDonors", "NumHAcceptors",
              "NumRings", "NumRotatableBonds", "NumHeavyAtoms",
              "NumHeteroatoms", "NumAromaticRings", "TPSA", "FractionCSP3"])


In [7]:
# Apply descriptor calculation to all SMILES
desc_df = smiles["SMILES"].apply(compute_descriptors)
desc_df

[08:57:27] Unusual charge on atom 42 number of radical electrons set to zero
[08:57:39] WARNING: not removing hydrogen atom without neighbors
[08:57:39] WARNING: not removing hydrogen atom without neighbors
[08:57:47] WARNING: not removing hydrogen atom without neighbors
[08:57:47] WARNING: not removing hydrogen atom without neighbors
[08:57:47] WARNING: not removing hydrogen atom without neighbors
[08:57:47] WARNING: not removing hydrogen atom without neighbors
[08:57:47] WARNING: not removing hydrogen atom without neighbors


,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,2180.317,-8.11643,28.0,29.0,6.0,66.0,155.0,57.0,3.0,901.57,0.540816
1,1269.433,-3.10570,17.0,16.0,6.0,31.0,91.0,32.0,4.0,495.89,0.508475
2,1811.253,4.86760,20.0,16.0,8.0,51.0,131.0,35.0,8.0,519.89,0.510417
3,1069.238,-4.13203,14.0,15.0,4.0,19.0,74.0,28.0,2.0,435.41,0.478261
4,1431.064,-0.50613,17.0,16.0,6.0,38.0,102.0,32.0,5.0,495.67,0.428571
...,...,...,...,...,...,...,...,...,...,...,...
12304,562.132,4.67384,2.0,7.0,5.0,7.0,38.0,10.0,2.0,92.89,0.571429
12305,489.495,1.07108,3.0,6.0,2.0,6.0,34.0,13.0,0.0,140.63,0.761905
12306,506.900,12.04840,0.0,2.0,0.0,30.0,36.0,2.0,0.0,26.30,0.911765
12307,450.792,10.48800,0.0,2.0,0.0,26.0,32.0,2.0,0.0,26.30,0.900000


In [8]:
# check invalid smiles
print(desc_df.isna().sum())

MolWt                0
LogP                 0
NumHDonors           0
NumHAcceptors        0
NumRings             0
NumRotatableBonds    0
NumHeavyAtoms        0
NumHeteroatoms       0
NumAromaticRings     0
TPSA                 0
FractionCSP3         0
dtype: int64


In [9]:
# Merge descriptors with original dataframe
data = pd.concat([vs_data, desc_df], axis=1)
data

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,2180.317,-8.11643,28.0,29.0,6.0,66.0,155.0,57.0,3.0,901.57,0.540816
1,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,1269.433,-3.10570,17.0,16.0,6.0,31.0,91.0,32.0,4.0,495.89,0.508475
2,Gramicidin D,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,1811.253,4.86760,20.0,16.0,8.0,51.0,131.0,35.0,8.0,519.89,0.510417
3,Desmopressin,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,1069.238,-4.13203,14.0,15.0,4.0,19.0,74.0,28.0,2.0,435.41,0.478261
4,Cetrorelix,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,1431.064,-0.50613,17.0,16.0,6.0,38.0,102.0,32.0,5.0,495.67,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12304,Tulmimetostat,COC1CN([C@H]2CC[C@H]([C@@]3(C)Oc4c(Cl)cc(C(=O)...,562.132,4.67384,2.0,7.0,5.0,7.0,38.0,10.0,2.0,92.89,0.571429
12305,Ibuzatrelvir,COC(=O)N[C@H](C(=O)N1C[C@H](C(F)(F)F)C[C@H]1C(...,489.495,1.07108,3.0,6.0,2.0,6.0,34.0,13.0,0.0,140.63,0.761905
12306,Cetyl oleate,CCCCCCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC,506.900,12.04840,0.0,2.0,0.0,30.0,36.0,2.0,0.0,26.30,0.911765
12307,Cetyl myristoleate,CCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC,450.792,10.48800,0.0,2.0,0.0,26.0,32.0,2.0,0.0,26.30,0.900000


In [10]:
# Drop any rows with invalid SMILES
data.dropna(inplace=True)
data

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3
0,Bivalirudin,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,2180.317,-8.11643,28.0,29.0,6.0,66.0,155.0,57.0,3.0,901.57,0.540816
1,Goserelin,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,1269.433,-3.10570,17.0,16.0,6.0,31.0,91.0,32.0,4.0,495.89,0.508475
2,Gramicidin D,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,1811.253,4.86760,20.0,16.0,8.0,51.0,131.0,35.0,8.0,519.89,0.510417
3,Desmopressin,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,1069.238,-4.13203,14.0,15.0,4.0,19.0,74.0,28.0,2.0,435.41,0.478261
4,Cetrorelix,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,1431.064,-0.50613,17.0,16.0,6.0,38.0,102.0,32.0,5.0,495.67,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12304,Tulmimetostat,COC1CN([C@H]2CC[C@H]([C@@]3(C)Oc4c(Cl)cc(C(=O)...,562.132,4.67384,2.0,7.0,5.0,7.0,38.0,10.0,2.0,92.89,0.571429
12305,Ibuzatrelvir,COC(=O)N[C@H](C(=O)N1C[C@H](C(F)(F)F)C[C@H]1C(...,489.495,1.07108,3.0,6.0,2.0,6.0,34.0,13.0,0.0,140.63,0.761905
12306,Cetyl oleate,CCCCCCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC,506.900,12.04840,0.0,2.0,0.0,30.0,36.0,2.0,0.0,26.30,0.911765
12307,Cetyl myristoleate,CCCC/C=C\CCCCCCCC(=O)OCCCCCCCCCCCCCCCC,450.792,10.48800,0.0,2.0,0.0,26.0,32.0,2.0,0.0,26.30,0.900000


In [11]:
## SAVE AS FINAL CSV FILE AS [ VS_curated_data.csv ]

data.to_csv("VS_curated_data.csv", index=False)

In [12]:
# 2 Compute Morgan fingerprints for the new compounds

# Use the same fingerprint generator as for training:

In [13]:
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# Load your saved Morgan generator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_morgan(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(1024)
    return morgan_gen.GetFingerprintAsNumPy(mol)

data['MorganFingerprint'] = data['SMILES'].apply(smiles_to_morgan)
X_vs = np.vstack(data['MorganFingerprint'].values)


[08:57:52] Unusual charge on atom 42 number of radical electrons set to zero
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors
[08:57:56] WARNING: not removing hydrogen atom without neighbors


In [15]:
# 3 Load your trained model

import joblib

model = joblib.load("rf_model.pkl")

In [16]:
# 4 Make predictions

# Predict probability of being active
y_proba = model.predict_proba(X_vs)[:, 1]  # probability of Active
data['Predicted_Prob'] = y_proba

# Optional: classify using 0.5 threshold
data['Predicted_Activity'] = (y_proba >= 0.5).astype(int)


In [17]:
# 5 Rank compounds

# You can now rank by predicted probability for prioritizing experimental validation:

data_sorted = data.sort_values(by='Predicted_Prob', ascending=False)
data_sorted.to_csv("virtual_screening_results.csv", index=False)
print(data_sorted)

               Common_Name                                             SMILES  \
11066          Ceralifimod  CCCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C...   
8271             Ponesimod  CCC/N=C1\S/C(=C\c2ccc(OC[C@H](O)CO)c(Cl)c2)C(=...   
10286            Etrasimod  O=C(O)C[C@H]1CCc2c1[nH]c1ccc(OCc3ccc(C4CCCC4)c...   
8563             Siponimod  CCc1cc(/C(C)=N/OCc2ccc(C3CCCCC3)c(C(F)(F)F)c2)...   
8762              Ozanimod  CC(C)Oc1ccc(-c2nc(-c3cccc4c3CC[C@@H]4NCCO)no2)...   
...                    ...                                                ...   
4191          Acetophenone                                     CC(=O)c1ccccc1   
1844                Isatin                                  O=C1Nc2ccccc2C1=O   
3666         Hypophosphite                                            O=P[O-]   
7707   Chromium picolinate  O=C([O-])c1ccccn1.O=C([O-])c1ccccn1.O=C([O-])c...   
2386    1,3-Dinitrobenzene                 O=[N+]([O-])c1cccc([N+](=O)[O-])c1   

         MolWt     LogP  Nu

In [18]:
data_sorted.head()

,Common_Name,SMILES,MolWt,LogP,NumHDonors,NumHAcceptors,NumRings,NumRotatableBonds,NumHeavyAtoms,NumHeteroatoms,NumAromaticRings,TPSA,FractionCSP3,MorganFingerprint,Predicted_Prob,Predicted_Activity
11066,Ceralifimod,CCCc1ccc(COc2ccc3c(c2)CCC(CN2CC(C(=O)O)C2)=C3C...,435.564,4.96280,1.0,4.0,4.0,9.0,32.0,5.0,2.0,59.00,0.444444,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1
8271,Ponesimod,CCC/N=C1\S/C(=C\c2ccc(OC[C@H](O)CO)c(Cl)c2)C(=...,460.983,4.26732,2.0,6.0,3.0,8.0,31.0,8.0,2.0,82.36,0.304348,"[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1
10286,Etrasimod,O=C(O)C[C@H]1CCc2c1[nH]c1ccc(OCc3ccc(C4CCCC4)c...,457.492,6.92780,2.0,2.0,5.0,6.0,33.0,7.0,3.0,62.32,0.423077,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1
8563,Siponimod,CCc1cc(/C(C)=N/OCc2ccc(C3CCCCC3)c(C(F)(F)F)c2)...,516.604,6.77270,1.0,4.0,4.0,9.0,37.0,8.0,2.0,62.13,0.517241,"[0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1
8762,Ozanimod,CC(C)Oc1ccc(-c2nc(-c3cccc4c3CC[C@@H]4NCCO)no2)...,404.470,3.63168,2.0,7.0,4.0,7.0,30.0,7.0,3.0,104.20,0.347826,"[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.0,1
